# Multi-node Dask

**Core notebook, 32-minute block with 18 minutes hands-on.** The notebook builds a task graph, a scheduler assigns ready tasks, and workers on Expanse compute nodes perform the calculations.

The key observation is that the Dask array expression is the same one used on a single node.

## 0. Start the cluster

In a JupyterLab terminal on the notebook node:

```bash
cd 6.1_python_for_HPC
bash dask_slurm/launch_scheduler.sh
```

In a separate Expanse login-node terminal:

```bash
cd 6.1_python_for_HPC
sbatch dask_slurm/dask_workers.slrm
squeue -u "$USER"
```

Full setup and cleanup instructions are in [`../dask_slurm/README.md`](../dask_slurm/README.md).

In [ ]:
!squeue --me

In [ ]:
import os
import socket
from pathlib import Path

from distributed import Client

scheduler_file = Path(
    os.environ.get("DASK_SCHEDULER_FILE", "~/.dask_scheduler.json")
).expanduser()

if not scheduler_file.exists():
    raise FileNotFoundError(
        f"Scheduler file not found: {scheduler_file}\n"
        "Start dask_slurm/launch_scheduler.sh in a JupyterLab terminal."
    )

client = Client(scheduler_file=str(scheduler_file), timeout="15s")

info = client.scheduler_info()
print("Notebook and scheduler host:", socket.gethostname())
print("Scheduler address:", client.scheduler.address)
print("Workers connected:", len(info["workers"]))
for address, worker in info["workers"].items():
    print(
        f"  {worker['name']} on {worker['host']}: "
        f"{worker['nthreads']} threads ({address})"
    )

Open the Dask dashboard through Jupyter Server Proxy:

<a href="/proxy/8787/status" target="_blank">Open the Dask dashboard</a>

Before continuing, identify the node running the notebook, the scheduler, and each worker.

## 1. Build an array across the workers

The classroom array represents about 12 GiB, divided into 128 MiB chunks. It is large enough to use multiple workers but small enough for a short exercise.

In [ ]:
import os
import time
import dask.array as da

test_mode = os.environ.get("PYHPC_TEST_MODE") == "1"
side = 8_000 if test_mode else 40_000
chunk_side = 4_000

array = da.ones((side, side), chunks=(chunk_side, chunk_side), dtype="float64")
expression = (
    array**2 + da.sin(array) * array * da.log(array)
).sum()

print("Shape:", array.shape)
print("Blocks:", array.numblocks)
print(f"Total size if fully stored: {array.nbytes / 1024**3:.2f} GiB")
print(f"Bytes per full chunk: {chunk_side**2 * 8 / 1024**2:.1f} MiB")

In [ ]:
started = time.perf_counter()
actual = expression.compute()
elapsed = time.perf_counter() - started

expected = float(side * side)
assert actual == expected
print(f"Verified result: {actual:,.0f}")
print(f"Elapsed: {elapsed:.3f} s")

## 2. Read the evidence

Use the dashboard or `client.scheduler_info()` to answer:

1. How many workers participated?
2. How many threads did each worker expose?
3. Did any worker approach its memory limit?
4. What changed in the Python expression when workers moved to more nodes?

In [ ]:
# Write your code here to inspect worker info.

<details><summary><b>Solution & Explanation</b></summary>

Expand below to inspect worker hosts, threads, and memory limits.
</details>

In [ ]:
final_info = client.scheduler_info()
{
    address: {
        "host": worker["host"],
        "threads": worker["nthreads"],
        "memory_limit_GiB": worker["memory_limit"] / 1024**3,
    }
    for address, worker in final_info["workers"].items()
}

## 3. Clean up

Close this client, cancel the worker job from the login node, and stop the scheduler with `Ctrl-C`:

```bash
scancel <worker_job_id>
squeue -u "$USER"
```

Do not leave worker jobs running after the exercise.

In [ ]:
client.close()
print("Notebook client closed. Cancel the worker job and stop the scheduler.")

## Takeaway

The same Dask calculation can run on workers on one or more nodes. Add nodes only after the one-node version gives the right answer, uses a useful chunk size, and has been timed.